# WAKIS FD – SIBC $D_{Z_s}$ Diagnose

Gleiche konforme STL-Geometrie für PEC und SIBC; Vergleich der elektrischen DOFs, SIBC-Diagonale und frequenzabhängigen Systemmatrix bei **1 GHz**. Benötigt die neue `solverFIT3D_FD.py` mit `_build_sibc_geometry` und den temporären PEC-artigen SIBC-Volumenmaterial-Workaround. Notebook aus dem Repository starten.

In [1]:
import numpy as np
import pyvista as pv
from pathlib import Path
from scipy.constants import pi
from tempfile import TemporaryDirectory

from wakis import GridFIT3D
from wakis.solverFIT3D_FD import SolverFIT3D_FD

In [2]:
# Identische Testgeometrie wie im bisherigen pytest-Regressionstest.
# Achtung: Diese Geometrie meldet ggf. einige Mehrfachschnitte.
tmp = TemporaryDirectory()
stl_path = Path(tmp.name) / "conductor.stl"

solid = pv.Cube(
    center=(0.08, -0.04, 0.03),
    x_length=0.97,
    y_length=0.85,
    z_length=0.92,
).triangulate()
solid.rotate_z(21.0, inplace=True)
solid.rotate_y(13.0, inplace=True)
solid.save(str(stl_path))


def make_solver(material_type):
    material = [np.inf, 1.0, 0.0] if material_type == "pec" else [1.0, 1.0, 5.8e7]
    grid = GridFIT3D(
        xmin=-1.0,
        xmax=1.0,
        ymin=-1.0,
        ymax=1.0,
        zmin=-1.0,
        zmax=1.0,
        Nx=12,
        Ny=12,
        Nz=12,
        stl_solids={"wall": str(stl_path)},
        stl_materials={"wall": material},
        stl_material_types={"wall": material_type},
        geometry_mode="conformal",
        stl_method="interior_points",
        verbose=0,
    )
    return SolverFIT3D_FD(
        grid=grid,
        use_stl=True,
        use_sibc=False,  # Alter TD-SIBC-Schalter bleibt aus
        bc_low=["PEC"] * 3,
        bc_high=["PEC"] * 3,
        verbose=0,
    )


solver_pec = make_solver("pec")
solver_sibc = make_solver("sibc")

print("PEC  active E DOFs:", len(solver_pec.free_E_idx))
print("SIBC active E DOFs:", len(solver_sibc.free_E_idx))
assert np.array_equal(solver_pec.free_E_idx, solver_sibc.free_E_idx), (
    "DOF-Listen weichen ab"
)

Generating grid with 1728 mesh cells...


/home/hschreib/Documents/Wakis/WAKIS/wakis/gridFIT3D.py:4434: UserWarning: Conformal geometry diagnostics:
10 edges contain geometric crossings that are hidden from endpoint topology and are ignored by the current conformal discretization.
10 edges contain multiple crossing positions. 0 of these are currently used by the conformal discretization; one representative crossing is selected on each edge.
10 edges cross the same STL at more than one distinct position. The geometry on these edges is not fully resolved by the current single-crossing treatment.
  primal x-edge (7, 4, 3): wall
  primal x-edge (7, 9, 3): wall
  primal y-edge (10, 4, 5): wall
  primal z-edge (3, 6, 4): wall
  dual y-edge (2, 6, 4): wall
  ... and 5 more.
  self._warn_interface_diagnostics()


Initializing Electromagnetic solver...
Generating grid with 1728 mesh cells...
Initializing Electromagnetic solver...
PEC  active E DOFs: 4425
SIBC active E DOFs: 4425


In [3]:
omega = 2 * pi * 1e9

print("SIBC active:", solver_sibc.has_sibc)
print("Number of SIBC magnetic DOFs:", np.count_nonzero(solver_sibc.sibc_g > 0))

D = solver_sibc._build_sibc_diagonal(omega)
mask = solver_sibc.sibc_g > 0

if np.any(mask):
    print("D_Zs real range:", D[mask].real.min(), D[mask].real.max())
    print("D_Zs imag range:", D[mask].imag.min(), D[mask].imag.max())
else:
    print("Keine aktiven SIBC-Flächen erkannt!")

A_pec = solver_pec.build_system_matrix(omega)
A_sibc = solver_sibc.build_system_matrix(omega)

difference = (A_sibc - A_pec).tocsr()
difference.eliminate_zeros()

print("Matrix shapes:", A_pec.shape, A_sibc.shape)
print("Nonzero matrix differences:", difference.nnz)
print(
    "Largest matrix difference:",
    np.max(np.abs(difference.data)) if difference.nnz else 0.0,
)

assert A_pec.shape == A_sibc.shape
assert solver_sibc.has_sibc and np.any(mask), "SIBC-Korrektur nicht aktiv"
assert difference.nnz > 0, "Keine Änderung der FD-Systemmatrix durch SIBC"

SIBC active: True
Number of SIBC magnetic DOFs: 488
D_Zs real range: 1.0000000856416962 1.0011597676506268
D_Zs imag range: -0.001159767650626804 -8.564169610926851e-08
Matrix shapes: (4425, 4425) (4425, 4425)
Nonzero matrix differences: 2282
Largest matrix difference: 6.438142755421844e-07


In [4]:
A_pec = solver_pec.build_system_matrix(omega)
A_sibc = solver_sibc.build_system_matrix(omega)

difference = (A_sibc - A_pec).tocsr()

rel_difference = np.linalg.norm(difference.data) / np.linalg.norm(A_pec.data)

print("Relative Frobenius norm difference:", rel_difference)

Relative Frobenius norm difference: 6.5917088582316115e-06


In [5]:
frequencies = np.array([1e8, 1e9, 1e10])

mask = solver_sibc.sibc_g > 0

for f in frequencies:
    omega_f = 2 * np.pi * f
    D = solver_sibc._build_sibc_diagonal(omega_f)

    correction = D[mask] - 1.0

    print(f"f = {f / 1e9:5.2f} GHz | max |D-1| = {np.max(np.abs(correction)):.8e}")

f =  0.10 GHz | max |D-1| = 5.18663861e-03
f =  1.00 GHz | max |D-1| = 1.64015914e-03
f = 10.00 GHz | max |D-1| = 5.18663861e-04


**Erwartung:** gleiche Matrixdimensionen, `has_sibc=True`, eine von eins verschiedene $D_{Z_s}$-Diagonale an SIBC-Flächen und eine nichtverschwindende Matrixdifferenz. Die bereits bekannten Mehrfachschnitt-Warnungen des STL-Testkörpers bleiben ein separates Geometrieproblem.